# DG-TSK: reproducing the article step by step

This notebook reproduces parts of Tables 2 and 3 of

> G. Xue, J. Wang, B. Zhang, B. Yuan and C. Dai, "Double groups of gates based
> Takagi-Sugeno-Kang (DG-TSK) fuzzy system for simultaneous feature selection and rule
> extraction", *Fuzzy Sets and Systems*, vol. 469, 2023.

It runs the same code as the script `examples/reproductions/dgtsk_2023.py`, one step at a
time:

1. settings and reproducibility;
2. the data;
3. one fold in detail: the rule base before training (Table 2) and DG-TSK (Table 3);
4. the full protocol of the article, ten-fold cross-validation repeated ten times;
5. the comparison with the article.

The full protocol takes about fifteen minutes on a laptop.

## 1. Settings and reproducibility

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

import highfis
from highfis import DGTSKClassifier
from highfis.memberships import GaussianMF
from highfis.models import TSKClassifierModel

The seed, the numbers of repetitions and folds, and the number of threads are fixed here.
The values of the article are copied from its Tables 2 and 3.

In [2]:
SEED = 0
REPETITIONS = 10
FOLDS = 10
THREADS = 1

DATASETS = {"Iris": load_iris, "Wine": load_wine}

# Values of the article, in percent: Table 2 (point-based rule base, no training) and
# Table 3 (DG-TSK: accuracy, selected features, extracted rules).
ARTICLE_TABLE_2 = {"Iris": 94.00, "Wine": 96.63}
ARTICLE_TABLE_3 = {"Iris": (96.8, 2.3, 3.1), "Wine": (98.3, 8.0, 3.0)}

Three lines make the run repeatable: a fixed number of threads, deterministic algorithms
in PyTorch, and no reuse of fuzzy sets between fits.

In [3]:
torch.set_num_threads(THREADS)
torch.use_deterministic_algorithms(True)
highfis.set_mf_cache_enabled(False)

## 2. The data

Iris and Wine ship with scikit-learn, so nothing is downloaded.

In [4]:
for name, load in DATASETS.items():
    features, labels = load(return_X_y=True)
    print(f"{name}: {features.shape[0]} samples, {features.shape[1]} features, {labels.max() + 1} classes")

Iris: 150 samples, 4 features, 3 classes
Wine: 178 samples, 13 features, 3 classes


## 3. One fold in detail

Before running the whole protocol, this section follows the first fold of Wine.

### The split and the scaling

The article scales the inputs to [0, 1]. The range is taken from the training part only.

In [5]:
features, labels = load_wine(return_X_y=True)
n_classes = int(labels.max()) + 1

folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
train, test = next(folds.split(features, labels))

scaler = MinMaxScaler().fit(features[train])
x_train = scaler.transform(features[train]).astype(np.float32)
x_test = scaler.transform(features[test]).astype(np.float32)
y_train, y_test = labels[train], labels[test]

print(f"training part: {x_train.shape[0]} samples, test part: {x_test.shape[0]} samples")

training part: 160 samples, test part: 18 samples


### Table 2: the rule base before training

The article starts from a rule base with **one rule per training sample**. For every
feature, the rule has a Gaussian fuzzy set centred on the value of the sample; all sets
share one spread, the mean of the standard deviations of the features (Eq. 23). The
consequent of the rule is the label of the sample. Nothing is trained.

In [6]:
spread = float(np.std(x_train, axis=0, ddof=1).mean())
fuzzy_sets = {
    f"x{d}": [GaussianMF(mean=float(centre), sigma=spread) for centre in x_train[:, d]]
    for d in range(x_train.shape[1])
}
rule_base = TSKClassifierModel(fuzzy_sets, n_classes=n_classes, rule_base="coco")
with torch.no_grad():
    rule_base.consequent_layer.weight.zero_()
    rule_base.consequent_layer.bias.copy_(torch.eye(n_classes)[torch.as_tensor(y_train)])
    predicted = rule_base(torch.as_tensor(x_test)).argmax(dim=1).numpy()

print(f"common spread: {spread:.3f}")
print(f"rules: {rule_base.n_rules}")
print(f"test accuracy without training: {100.0 * np.mean(predicted == y_test):.2f}%")

common spread: 0.201
rules: 160
test accuracy without training: 100.00%


### Table 3: DG-TSK

DG-TSK trains two groups of gates on that rule base, one gate per feature and one per
rule, removes the features and rules whose gates stay closed, and fine-tunes what
remains. The defaults of `DGTSKClassifier` are the settings of the article, so the model
is built without arguments other than the seed.

In [7]:
model = DGTSKClassifier(random_state=SEED).fit(x_train, y_train)

print(f"test accuracy: {100.0 * model.score(x_test, y_test):.2f}%")
print(f"selected features: {len(model.selected_features_)} of {x_train.shape[1]}")
print(f"extracted rules: {model.model_.n_rules} of {x_train.shape[0]}")

test accuracy: 100.00%
selected features: 5 of 13
extracted rules: 3 of 160


The selected features, by their names in the dataset:

In [8]:
feature_names = load_wine().feature_names
print([feature_names[d] for d in model.selected_features_])

['alcohol', 'flavanoids', 'color_intensity', 'od280/od315_of_diluted_wines', 'proline']


## 4. The full protocol

The article reports the mean over ten-fold cross-validation repeated ten times, that is,
100 runs per dataset. The loop below repeats the two steps of the previous section for
every run and stores the accuracy and, for DG-TSK, the numbers of selected features and
extracted rules.

In [9]:
untrained = {name: [] for name in DATASETS}
trained = {name: [] for name in DATASETS}

for name, load in DATASETS.items():
    features, labels = load(return_X_y=True)
    n_classes = int(labels.max()) + 1

    for repetition in range(REPETITIONS):
        folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED + repetition)
        for train, test in folds.split(features, labels):
            scaler = MinMaxScaler().fit(features[train])
            x_train = scaler.transform(features[train]).astype(np.float32)
            x_test = scaler.transform(features[test]).astype(np.float32)
            y_train, y_test = labels[train], labels[test]

            # Table 2: the point-based rule base, without training.
            spread = float(np.std(x_train, axis=0, ddof=1).mean())
            fuzzy_sets = {
                f"x{d}": [GaussianMF(mean=float(centre), sigma=spread) for centre in x_train[:, d]]
                for d in range(x_train.shape[1])
            }
            rule_base = TSKClassifierModel(fuzzy_sets, n_classes=n_classes, rule_base="coco")
            with torch.no_grad():
                rule_base.consequent_layer.weight.zero_()
                rule_base.consequent_layer.bias.copy_(torch.eye(n_classes)[torch.as_tensor(y_train)])
                predicted = rule_base(torch.as_tensor(x_test)).argmax(dim=1).numpy()
            untrained[name].append(100.0 * np.mean(predicted == y_test))

            # Table 3: DG-TSK with its defaults.
            model = DGTSKClassifier(random_state=SEED + repetition).fit(x_train, y_train)
            trained[name].append(
                (100.0 * model.score(x_test, y_test), len(model.selected_features_), model.model_.n_rules)
            )
    print(f"{name}: {len(trained[name])} runs done")

Iris: 100 runs done


Wine: 100 runs done


## 5. Comparison with the article

### Table 2: point-based rule base without training

In [10]:
print("Table 2: point-based rule base without training (accuracy in percent)")
print(f"{'dataset':8s} {'article':>8s} {'highFIS':>8s}")
for name in DATASETS:
    print(f"{name:8s} {ARTICLE_TABLE_2[name]:8.2f} {np.mean(untrained[name]):8.2f}")

Table 2: point-based rule base without training (accuracy in percent)
dataset   article  highFIS
Iris        94.00    95.47
Wine        96.63    96.58


### Table 3: DG-TSK

In [11]:
print("Table 3: DG-TSK (accuracy in percent / selected features / extracted rules)")
print(f"{'dataset':8s} {'article':>18s} {'highFIS':>18s} {'std of accuracy':>16s}")
for name in DATASETS:
    accuracy, n_features, n_rules = np.asarray(trained[name]).T
    reported = "{:.1f} / {:.1f} / {:.1f}".format(*ARTICLE_TABLE_3[name])
    measured = f"{accuracy.mean():.1f} / {n_features.mean():.1f} / {n_rules.mean():.1f}"
    print(f"{name:8s} {reported:>18s} {measured:>18s} {accuracy.std():16.2f}")

Table 3: DG-TSK (accuracy in percent / selected features / extracted rules)
dataset             article            highFIS  std of accuracy
Iris       96.8 / 2.3 / 3.1   95.7 / 2.0 / 6.0             5.29
Wine       98.3 / 8.0 / 3.0   97.7 / 5.1 / 3.0             3.37


The accuracies are within about one point of the article, well inside the standard
deviation between runs. The structure differs on two points: on Iris highFIS keeps six
rules where the article reports three, and on Wine it keeps five features where the
article reports eight. The documentation page of this reproduction discusses both and
compares them with the reference code of the authors.

## Environment

The versions that produced the numbers above:

In [12]:
highfis.show_versions()

highfis       : 0.33.0
python        : 3.11.15
platform      : Linux-7.0.0-38-generic-x86_64-with-glibc2.43
torch         : 2.14.1+cu130
numpy         : 2.4.6
scikit-learn  : 1.9.1
default dtype : torch.float32
threads       : 1
cpu           : Intel(R) Core(TM) Ultra 7 258V
